# Context Window Advantage Experiment - Minimal Pipeline Demo

This notebook demonstrates the core pipeline for testing whether models with larger maximum context windows outperform models with smaller maximum context windows when both operate at identical context window sizes.

**Research Question**: Does GPT-4 Turbo (128K capacity) outperform GPT-4 (8K capacity) when both process the same smaller context (2K-6K tokens)?

## Methodology Overview

1. **Load Fixed Q&As**: Use pre-generated question-answer pairs for consistent evaluation
2. **Build Haystacks**: Create documents of precise token lengths using real Paul Graham essays and ArXiv papers
3. **Insert Needles**: Place the facts (answers) that need to be retrieved at different positions
4. **Query Models**: Test both high-capacity and low-capacity models on identical contexts
5. **Evaluate**: Measure retrieval accuracy and analyze performance patterns

## Data Sources
- **Fixed Q&As**: 40 pre-generated questions across 4 complexity types
- **Documents**: Real Paul Graham essays and ArXiv papers from project data directory
- **Models**: GPT-4 (8K) vs GPT-4 Turbo (128K) via ORQ API

## 1. Setup and Imports

Import the real project components for haystack building, needle generation, model interface, and evaluation.

In [3]:
%load_ext autoreload
%autoreload 2

import sys
import os
import json
import random
from pathlib import Path
from typing import Dict, List, Any
from dataclasses import dataclass, asdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from rich.console import Console
from rich.progress import track
import tiktoken

# Add project root to path for imports
project_root = Path.cwd().parent.parent
sys.path.append(str(project_root))

# Import real project components
from experiments.context_advantage.needle_haystack.haystack_builder import HaystackBuilder, Haystack
from experiments.context_advantage.needle_haystack.needle_generator import NeedleGenerator, FixedQuestionAnswer
from context_is_king.models import ModelInterface, QueryResult
from context_is_king.evaluation import NeedleEvaluator

# Initialize rich console for pretty printing
console = Console()

# Load environment variables
import dotenv
dotenv.load_dotenv(dotenv_path=project_root / ".env", override=True)

console.print("✅ Imported all real project components", style="green")
console.print(f"📁 Project root: {project_root}", style="blue")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


✅ Imported all real project components

📁 Project root: /Users/baukebrenninkmeijer/Developer/pydata-2025-context-is-king

## 2. Configuration

Set up the experiment parameters using real model configurations and realistic context sizes.

In [4]:
# Experiment Configuration
@dataclass
class ExperimentConfig:
    # Model comparison: High capacity vs Low capacity
    high_capacity_model: str = "gpt-4-turbo"  # 128K context window
    low_capacity_model: str = "gpt-4"  # 8K context window

    # Context sizes to test (within both models' capabilities)
    context_sizes: list[int] = None

    # Document composition strategy
    composition: str = "mixed"  # "pg_heavy", "arxiv_heavy", or "mixed"

    # Number of questions per type for demo (total = 4 * num_per_type)
    num_questions_per_type: int = 2

    # Needle insertion positions
    needle_positions: list[str] = None

    # Model query parameters
    temperature: float = 0.0  # Deterministic responses
    max_tokens: int = 100


# Initialize configuration
config = ExperimentConfig(
    context_sizes=[2000, 4000, 6000],  # Progressive sizes within both models' range
    needle_positions=["beginning", "center", "end"],
)

console.print("⚙️ Experiment Configuration:", style="bold blue")
console.print(f"   High Capacity: {config.high_capacity_model}")
console.print(f"   Low Capacity: {config.low_capacity_model}")
console.print(f"   Context Sizes: {config.context_sizes}")
console.print(f"   Composition: {config.composition}")
console.print(f"   Questions per type: {config.num_questions_per_type}")
console.print(f"   Total questions: {4 * config.num_questions_per_type}")

⚙️ Experiment Configuration:

High Capacity: gpt-4-turbo

Low Capacity: gpt-4

Context Sizes: [2000, 4000, 6000]

Composition: mixed

Questions per type: 2

Total questions: 8

## 3. Load Fixed Q&As and Generate Needles

Load the pre-generated question-answer pairs that ensure consistent evaluation across different runs and models.

In [5]:
# Initialize needle generator with fixed Q&As
fixed_qa_dir = Path("data/fixed_qas")
needle_generator = NeedleGenerator(use_fixed_qas=True, fixed_qa_dir=fixed_qa_dir)

console.print(f"📚 Loaded fixed Q&As from: {fixed_qa_dir.absolute()}", style="green")

# Select subset of questions for demo
selected_qas = []
question_types = ["direct", "cross_reference", "synthesis", "domain_transfer"]

for qtype in question_types:
    if qtype in needle_generator.fixed_qas:
        # Take first N questions of each type
        type_qas = needle_generator.fixed_qas[qtype][:config.num_questions_per_type]
        selected_qas.extend(type_qas)
        console.print(f"   📝 {qtype}: {len(type_qas)} questions")
    else:
        console.print(f"   ⚠️  {qtype}: No questions found", style="yellow")

console.print(f"\n✅ Total selected questions: {len(selected_qas)}")

# Extract the needles (facts/answers) that need to be inserted into haystacks
needles_from_qas = []
for qa in selected_qas:
    needles_from_qas.append({
        'content': qa.answer,
        'question': qa.question,
        'type': qa.question_type,
        'id': qa.id
    })

# Preview the needles
console.print("\n🎯 Sample Needles:", style="bold blue")
for i, needle in enumerate(needles_from_qas[:3]):
    console.print(f"   {i+1}. Type: {needle['type']}")
    console.print(f"      Question: {needle['question'][:80]}...")
    console.print(f"      Answer: {needle['content'][:60]}...\n")

📚 Loaded 10 fixed direct Q&As
📚 Loaded 10 fixed cross_reference Q&As
📚 Loaded 10 fixed synthesis Q&As
📚 Loaded 10 fixed domain_transfer Q&As
🧵 Initialized Needle Generator with fixed Q&As from data/fixed_qas


📚 Loaded fixed Q&As from: 
/Users/baukebrenninkmeijer/Developer/pydata-2025-context-is-king/experiments/context_advantage/data/fixed_qas

📝 direct: 2 questions

📝 cross_reference: 2 questions

📝 synthesis: 2 questions

...

Question: What is the founding year of Aurora Technologies?...

Answer: Aurora Technologies was founded in 2003....

3. Type: cross_reference

Question: Which company founded in 2005 has its headquarters in Seattle?...

Answer: Nexus Innovations was founded in 2005 and has its headquarte...

## 4. Build Haystacks with Real Documents

Create documents of precise token lengths using the real HaystackBuilder and actual Paul Graham essays + ArXiv papers.

In [8]:
# Initialize haystack builder with real document data
console.print("🏗️ Initializing HaystackBuilder...", style="blue")
haystack_builder = HaystackBuilder()

console.print(f"📚 Available documents:")
console.print(f"   Paul Graham essays: {len(haystack_builder.pg_documents)}")
console.print(f"   ArXiv papers: {len(haystack_builder.arxiv_documents)}")

# Build haystacks at different context sizes
haystacks = {}
haystack_metadata = {}

for size in track(config.context_sizes, description="Building haystacks"):
    console.print(f"\n🔨 Building {size}-token haystack with {config.composition} composition")

    # Build base haystack
    haystack = haystack_builder.build_haystack(
        target_size=size,
        composition=config.composition,
        shuffle=False,  # Keep deterministic for demo
    )

    console.print(f"   ✅ Built haystack: {haystack.token_count} tokens (target: {size})")
    console.print(f"   📄 Documents used: {len(haystack.documents_used)}")

    haystacks[size] = haystack.content
    haystack_metadata[size] = {
        "actual_tokens": haystack.token_count,
        "target_tokens": size,
        "documents_used": haystack.documents_used,
        "composition": haystack.composition,
    }

console.print("\n✅ All haystacks built successfully!", style="green")

🏗️ Initializing HaystackBuilder...

📚 Loaded 64 PG essays, 100 ArXiv papers


📚 Available documents:

Paul Graham essays: 64

ArXiv papers: 100

...

🏗️  Building mixed haystack targeting 6,000 tokens...

✅ Built haystack: 5,590 tokens (81 documents)

✅ Built haystack: 5590 tokens (target: 6000)

📄 Documents used: 81

✅ All haystacks built successfully!

## 5. Insert Needles into Haystacks

Insert the needle facts at strategic positions within each haystack to test retrieval across different locations.

In [9]:
def insert_needles_at_positions(haystack_content: str, needles: List[Dict], positions: List[str]) -> tuple[str, Dict]:
    """
    Insert needles at specified positions in the haystack.
    Returns modified haystack and position tracking info.
    """
    encoding = tiktoken.encoding_for_model("gpt-4o")
    tokens = encoding.encode(haystack_content)
    total_tokens = len(tokens)

    # Calculate insertion positions
    position_map = {
        "beginning": int(0.05 * total_tokens),  # 5% from start
        "center": int(0.5 * total_tokens),  # Middle
        "end": int(0.95 * total_tokens),  # 5% from end
    }

    # Sort positions by location (end to beginning to maintain token positions)
    sorted_positions = sorted(positions, key=lambda p: position_map[p], reverse=True)

    modified_content = haystack_content
    needle_positions = {}

    # Insert needles (working backwards to maintain positions)
    for i, pos_name in enumerate(sorted_positions):
        if i < len(needles):
            needle = needles[i]
            insertion_point = position_map[pos_name]

            # Convert token position to character position (approximate)
            char_pos = int(insertion_point * len(modified_content) / total_tokens)

            # Insert needle with some context formatting
            needle_text = f"\n\n{needle['content']}\n\n"
            modified_content = modified_content[:char_pos] + needle_text + modified_content[char_pos:]

            needle_positions[needle["id"]] = {
                "position": pos_name,
                "token_pos": insertion_point,
                "char_pos": char_pos,
                "content": needle["content"],
                "question": needle["question"],
            }

    return modified_content, needle_positions


# Insert needles into each haystack
haystacks_with_needles = {}
needle_position_tracking = {}

for size, haystack_content in haystacks.items():
    console.print(f"\n🎯 Inserting needles into {size}-token haystack")

    # Select needles for this haystack (rotate through available needles)
    needles_for_haystack = needles_from_qas[: len(config.needle_positions)]

    modified_haystack, positions = insert_needles_at_positions(
        haystack_content, needles_for_haystack, config.needle_positions
    )

    haystacks_with_needles[size] = modified_haystack
    needle_position_tracking[size] = positions

    console.print(f"   📌 Inserted {len(positions)} needles at positions:")
    for needle_id, pos_info in positions.items():
        console.print(f"      {needle_id}: {pos_info['position']} (token ~{pos_info['token_pos']})")

console.print("\n✅ Needles inserted into all haystacks!", style="green")

🎯 Inserting needles into 2000-token haystack

📌 Inserted 3 needles at positions:

direct_01: end (token ~1845)

direct_02: center (token ~971)

cross_ref_01: beginning (token ~97)

...

📌 Inserted 3 needles at positions:

direct_01: end (token ~5310)

direct_02: center (token ~2795)

cross_ref_01: beginning (token ~279)

✅ Needles inserted into all haystacks!

## 6. Query Models with Real ORQ API

Test both high-capacity and low-capacity models on identical haystacks using the real ModelInterface and ORQ API.

In [ ]:
# Initialize model interface
console.print("🤖 Initializing ModelInterface with ORQ API...", style="blue")
model_interface = ModelInterface()

# Get model configurations for context window info
high_cap_config = model_interface.MODELS[config.high_capacity_model]
low_cap_config = model_interface.MODELS[config.low_capacity_model]

console.print(f"📊 Model Comparison:")
console.print(f"   High Capacity: {high_cap_config.name} ({high_cap_config.max_context_tokens:,} tokens)")
console.print(f"   Low Capacity: {low_cap_config.name} ({low_cap_config.max_context_tokens:,} tokens)")
console.print(f"   Cost per 1K tokens: ${high_cap_config.cost_per_1k_tokens} vs ${low_cap_config.cost_per_1k_tokens}")

# Test API connection
console.print("\n🔌 Testing API connections...")
test_result = model_interface.test_connection(config.high_capacity_model)
if test_result:
    console.print("   ✅ API connection successful", style="green")
else:
    console.print("   ❌ API connection failed", style="red")
    console.print("   Please check your ORQ_API_KEY environment variable")

In [ ]:
# Query both models on all haystacks
results = {}
total_cost = 0.0

models_to_test = [config.high_capacity_model, config.low_capacity_model]

for model_name in models_to_test:
    console.print(f"\n🚀 Querying {model_name}...", style="bold blue")
    results[model_name] = {}
    
    for context_size in config.context_sizes:
        console.print(f"\n   📏 Context size: {context_size} tokens")
        haystack_content = haystacks_with_needles[context_size]
        positions = needle_position_tracking[context_size]
        
        responses = []
        
        # Query with each question that has a corresponding needle in this haystack
        for needle_id, pos_info in track(positions.items(), description=f"Querying {model_name}"):
            question = pos_info['question']
            expected_answer = pos_info['content']
            
            # Construct prompt
            prompt = f"""{haystack_content}

Based on the text above, please answer the following question:
Question: {question}

Answer:"""
            
            # Query model
            try:
                result = model_interface.query_model(
                    model_name=model_name,
                    prompt=prompt,
                    max_tokens=config.max_tokens,
                    temperature=config.temperature
                )
                
                if result.success:
                    total_cost += result.estimated_cost
                    console.print(f"      ✅ {needle_id}: ${result.estimated_cost:.4f}")
                else:
                    console.print(f"      ❌ {needle_id}: {result.error_message}", style="red")
                
                responses.append({
                    'needle_id': needle_id,
                    'question': question,
                    'expected_answer': expected_answer,
                    'actual_response': result.response if result.success else "",
                    'success': result.success,
                    'error': result.error_message,
                    'position': pos_info['position'],
                    'response_time': result.response_time,
                    'cost': result.estimated_cost,
                    'tokens_used': result.total_tokens
                })
                
            except Exception as e:
                console.print(f"      💥 {needle_id}: Exception - {str(e)}", style="red")
                responses.append({
                    'needle_id': needle_id,
                    'question': question,
                    'expected_answer': expected_answer,
                    'actual_response': "",
                    'success': False,
                    'error': str(e),
                    'position': pos_info['position'],
                    'response_time': 0,
                    'cost': 0,
                    'tokens_used': 0
                })
        
        results[model_name][context_size] = responses

console.print(f"\n💰 Total experiment cost: ${total_cost:.2f}", style="green")
console.print("✅ All model queries completed!", style="green")

## 7. Evaluate Performance with Real NeedleEvaluator

Use the project's NeedleEvaluator to score retrieval accuracy and analyze performance patterns.

In [ ]:
# Initialize evaluator
console.print("📊 Evaluating model performance...", style="blue")
evaluator = NeedleEvaluator()

# Calculate performance metrics
performance_summary = {}
detailed_scores = {}

for model_name, model_results in results.items():
    console.print(f"\n🎯 Evaluating {model_name}:")
    performance_summary[model_name] = {}
    detailed_scores[model_name] = {}
    
    for context_size, responses in model_results.items():
        successful_responses = [r for r in responses if r['success']]
        
        if not successful_responses:
            console.print(f"   ⚠️  {context_size} tokens: No successful responses", style="yellow")
            performance_summary[model_name][context_size] = {
                'accuracy': 0.0,
                'avg_score': 0.0,
                'success_rate': 0.0,
                'total_responses': len(responses)
            }
            continue
        
        # Calculate retrieval scores using real evaluator
        scores = []
        position_scores = {pos: [] for pos in config.needle_positions}
        
        for response in successful_responses:
            try:
                # Use real NeedleEvaluator scoring
                score = evaluator.score_needle_retrieval(
                    expected=response['expected_answer'],
                    actual=response['actual_response'],
                    question=response['question']
                )
                scores.append(score)
                position_scores[response['position']].append(score)
                
            except Exception as e:
                console.print(f"   ⚠️  Scoring error for {response['needle_id']}: {e}", style="yellow")
                scores.append(0.0)
        
        # Calculate summary statistics
        avg_score = np.mean(scores) if scores else 0.0
        accuracy = sum(1 for s in scores if s > 0.8) / len(scores) if scores else 0.0
        success_rate = len(successful_responses) / len(responses)
        
        performance_summary[model_name][context_size] = {
            'accuracy': accuracy,
            'avg_score': avg_score,
            'success_rate': success_rate,
            'total_responses': len(responses),
            'successful_responses': len(successful_responses)
        }
        
        detailed_scores[model_name][context_size] = {
            'individual_scores': scores,
            'position_scores': position_scores
        }
        
        console.print(f"   📏 {context_size} tokens: Avg Score: {avg_score:.3f}, Accuracy: {accuracy:.3f}, Success Rate: {success_rate:.3f}")

console.print("\n✅ Performance evaluation completed!", style="green")

## 8. Visualization and Results Analysis

Visualize the performance comparison and analyze whether high-capacity models outperform low-capacity models at identical context sizes.

In [ ]:
# Create performance comparison visualizations
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Context Window Advantage Analysis', fontsize=16, fontweight='bold')

# Prepare data for plotting
models = list(performance_summary.keys())
context_sizes = config.context_sizes

# Colors for models
colors = {'gpt-4-turbo': '#1f77b4', 'gpt-4': '#ff7f0e'}
model_colors = [colors.get(model, '#2ca02c') for model in models]

# 1. Accuracy by Context Size
for i, model in enumerate(models):
    accuracies = [performance_summary[model][size]['accuracy'] for size in context_sizes]
    ax1.plot(context_sizes, accuracies, marker='o', linewidth=2, 
             color=model_colors[i], label=f"{model} ({model_interface.MODELS[model].max_context_tokens//1000}K max)")

ax1.set_xlabel('Context Size (tokens)')
ax1.set_ylabel('Accuracy (score > 0.8)')
ax1.set_title('Accuracy vs Context Size')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 2. Average Score by Context Size
for i, model in enumerate(models):
    avg_scores = [performance_summary[model][size]['avg_score'] for size in context_sizes]
    ax2.plot(context_sizes, avg_scores, marker='s', linewidth=2,
             color=model_colors[i], label=model)

ax2.set_xlabel('Context Size (tokens)')
ax2.set_ylabel('Average Score')
ax2.set_title('Average Retrieval Score vs Context Size')
ax2.legend()
ax2.grid(True, alpha=0.3)

# 3. Performance by Position (for largest context size)
largest_context = max(context_sizes)
positions = config.needle_positions
x_pos = np.arange(len(positions))
width = 0.35

for i, model in enumerate(models):
    if largest_context in detailed_scores[model]:
        pos_scores = detailed_scores[model][largest_context]['position_scores']
        pos_avg = [np.mean(pos_scores[pos]) if pos_scores[pos] else 0 for pos in positions]
        ax3.bar(x_pos + i*width, pos_avg, width, label=model, color=model_colors[i], alpha=0.8)

ax3.set_xlabel('Needle Position')
ax3.set_ylabel('Average Score')
ax3.set_title(f'Performance by Position ({largest_context} tokens)')
ax3.set_xticks(x_pos + width/2)
ax3.set_xticklabels(positions)
ax3.legend()
ax3.grid(True, alpha=0.3)

# 4. Cost-Performance Analysis
for model in models:
    model_config = model_interface.MODELS[model]
    accuracies = [performance_summary[model][size]['accuracy'] for size in context_sizes]
    
    # Estimate cost per query (rough approximation)
    avg_tokens_per_query = np.mean(context_sizes) + config.max_tokens
    cost_per_query = (avg_tokens_per_query / 1000) * model_config.cost_per_1k_tokens
    
    # Plot cost-effectiveness (accuracy per dollar)
    cost_effectiveness = [acc / cost_per_query if cost_per_query > 0 else 0 for acc in accuracies]
    
    ax4.plot(context_sizes, cost_effectiveness, marker='d', linewidth=2,
             label=f"{model} (${cost_per_query:.4f}/query)")

ax4.set_xlabel('Context Size (tokens)')
ax4.set_ylabel('Accuracy per Dollar')
ax4.set_title('Cost-Effectiveness Analysis')
ax4.legend()
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 9. Summary and Statistical Analysis

Summarize the findings and determine whether high-capacity models demonstrate superior performance at identical context sizes.

In [ ]:
# Create comprehensive summary
console.print("\n" + "="*70, style="bold blue")
console.print("CONTEXT WINDOW ADVANTAGE ANALYSIS - RESULTS SUMMARY", style="bold blue")
console.print("="*70, style="bold blue")

# Model comparison overview
console.print(f"\n📊 Models Compared:")
for model in models:
    config = model_interface.MODELS[model]
    console.print(f"   • {config.name}: {config.max_context_tokens:,} token capacity, ${config.cost_per_1k_tokens}/1K tokens")

console.print(f"\n🎯 Test Configuration:")
console.print(f"   • Context sizes tested: {config.context_sizes}")
console.print(f"   • Questions per type: {config.num_questions_per_type}")
console.print(f"   • Total questions: {len(selected_qas)}")
console.print(f"   • Document composition: {config.composition}")

# Performance comparison table
console.print(f"\n📈 Performance Results:")
performance_df = pd.DataFrame({
    f"{model}_accuracy": [performance_summary[model][size]['accuracy'] for size in context_sizes]
    for model in models
}, index=[f"{size} tokens" for size in context_sizes])

# Add average scores
for model in models:
    performance_df[f"{model}_avg_score"] = [performance_summary[model][size]['avg_score'] for size in context_sizes]

console.print(performance_df.round(3))

# Statistical significance analysis (if we have enough data)
console.print(f"\n📊 Statistical Analysis:")

if len(models) == 2:
    model1, model2 = models
    
    # Compare overall performance
    all_scores_m1 = []
    all_scores_m2 = []
    
    for size in context_sizes:
        if size in detailed_scores[model1] and size in detailed_scores[model2]:
            all_scores_m1.extend(detailed_scores[model1][size]['individual_scores'])
            all_scores_m2.extend(detailed_scores[model2][size]['individual_scores'])
    
    if all_scores_m1 and all_scores_m2:
        from scipy import stats
        
        # Perform t-test
        t_stat, p_value = stats.ttest_ind(all_scores_m1, all_scores_m2)
        
        console.print(f"   • {model1} mean score: {np.mean(all_scores_m1):.3f} (n={len(all_scores_m1)})")
        console.print(f"   • {model2} mean score: {np.mean(all_scores_m2):.3f} (n={len(all_scores_m2)})")
        console.print(f"   • t-statistic: {t_stat:.3f}")
        console.print(f"   • p-value: {p_value:.3f}")
        
        if p_value < 0.05:
            winner = model1 if np.mean(all_scores_m1) > np.mean(all_scores_m2) else model2
            console.print(f"   • 🏆 Statistically significant advantage: {winner}", style="green")
        else:
            console.print(f"   • 📊 No statistically significant difference (p > 0.05)", style="yellow")

# Key findings
console.print(f"\n🔍 Key Findings:")

# Find which model performs better at each context size
for size in context_sizes:
    scores_by_model = {model: performance_summary[model][size]['accuracy'] for model in models}
    best_model = max(scores_by_model, key=scores_by_model.get)
    best_score = scores_by_model[best_model]
    
    console.print(f"   • At {size} tokens: {best_model} leads with {best_score:.3f} accuracy")

# Overall conclusion
if len(models) == 2:
    high_cap_model = config.high_capacity_model
    low_cap_model = config.low_capacity_model
    
    high_cap_avg = np.mean([performance_summary[high_cap_model][size]['accuracy'] for size in context_sizes])
    low_cap_avg = np.mean([performance_summary[low_cap_model][size]['accuracy'] for size in context_sizes])
    
    console.print(f"\n🎯 CONCLUSION:", style="bold green")
    console.print(f"   Average Accuracy - {high_cap_model}: {high_cap_avg:.3f}")
    console.print(f"   Average Accuracy - {low_cap_model}: {low_cap_avg:.3f}")
    
    if high_cap_avg > low_cap_avg + 0.05:  # 5% threshold
        console.print(f"   ✅ {high_cap_model} outperforms {low_cap_model} at identical context sizes", style="green")
        console.print(f"   💡 High-capacity models may have superior attention mechanisms", style="blue")
    elif low_cap_avg > high_cap_avg + 0.05:
        console.print(f"   ✅ {low_cap_model} outperforms {high_cap_model} at identical context sizes", style="green")
        console.print(f"   💡 Specialized models may excel within their optimal range", style="blue")
    else:
        console.print(f"   📊 Performance is similar between models", style="yellow")
        console.print(f"   💡 Context window capacity may not significantly impact sub-capacity performance", style="blue")

console.print(f"\n💰 Total Experiment Cost: ${total_cost:.2f}", style="green")
console.print("="*70, style="bold blue")

## 10. Export Results

Save the experimental results for further analysis and comparison with the full experiment.

In [ ]:
# Export results to JSON for further analysis
import datetime

# Prepare export data
export_data = {
    'experiment_info': {
        'timestamp': datetime.datetime.now().isoformat(),
        'experiment_type': 'minimal_context_advantage_demo',
        'models_tested': models,
        'context_sizes': config.context_sizes,
        'total_questions': len(selected_qas),
        'total_cost': total_cost
    },
    'model_configs': {
        model: asdict(model_interface.MODELS[model]) for model in models
    },
    'performance_summary': performance_summary,
    'detailed_results': results,
    'haystack_metadata': haystack_metadata,
    'needle_positions': needle_position_tracking
}

# Save to file
results_dir = Path("results")
results_dir.mkdir(exist_ok=True)

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
results_file = results_dir / f"minimal_demo_results_{timestamp}.json"

with open(results_file, 'w') as f:
    json.dump(export_data, f, indent=2, default=str)

console.print(f"\n💾 Results exported to: {results_file.absolute()}", style="green")

# Also save a summary CSV for easy analysis
summary_csv = results_dir / f"performance_summary_{timestamp}.csv"
performance_df.to_csv(summary_csv)

console.print(f"📊 Performance summary saved to: {summary_csv.absolute()}", style="green")

console.print("\n✅ Minimal Context Advantage Experiment Complete!", style="bold green")
console.print("🎯 This demo shows the core pipeline using real project components.", style="blue")
console.print("📈 Scale up to full experiment for comprehensive statistical analysis.", style="blue")